# 🎶 04 – Analyse de signaux avec SciPy

---

## 🎯 Objectifs
- Comprendre ce qu'est un **signal** et pourquoi on l'analyse
- Décomposer un signal en fréquences avec la **FFT** (sans formule)
- Appliquer la FFT à des cas concrets : son, capteur, données temporelles
- **Filtrer** un signal bruité pour retrouver l'information utile
- Détecter des **anomalies** et des **cycles** dans des données réelles

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import butter, filtfilt
print("SciPy Signal prêt ✅")

---
## 📝 Partie 1 – Qu'est-ce qu'un signal ?

### 🔎 Définition en langage humain

Un **signal** est une mesure qui **varie dans le temps**. C'est la forme la plus courante de données dans le monde réel :

| Signal | Ce qui varie | À quelle fréquence |
|--------|-------------|-------------------|
| Son / musique | Pression de l'air | 20 à 20 000 fois/seconde |
| Électrocardiogramme | Activité électrique du cœur | ~1 fois/seconde |
| Température usine | Température (°C) | Toutes les minutes |
| Ventes quotidiennes | CA en € | 1 fois/jour |
| Vibrations machine | Amplitude (mm) | Des milliers de fois/seconde |

### 🔎 Le problème des signaux réels : le bruit

Dans la réalité, les signaux ne sont **jamais parfaits**. Il y a toujours du **bruit** — des variations parasites qui masquent l'information utile.

```
Signal idéal :          Signal réel (bruité) :

    ╭───╮                   ╭─╮ ╭╮
   ╱     ╲               ╭╯  ╰╯ ╰╮
──╯         ╰──         ─╯       ╰─╯╲╭
                                      ╰

→ La forme de fond reste visible, mais les variations parasites gênent
```

**L'analyse de signal** sert à :
1. **Décomposer** un signal pour voir quelles fréquences le composent (FFT)
2. **Filtrer** pour éliminer le bruit et garder l'information utile
3. **Détecter** des anomalies, des cycles, des tendances

### 🔎 Notion clé : la fréquence d'échantillonnage

Pour travailler avec un signal numérique, on le **mesure à intervalles réguliers**. La **fréquence d'échantillonnage** `fs` indique combien de mesures on prend par seconde.

```
fs = 1000 Hz → on mesure 1000 fois par seconde
fs = 10 Hz   → on mesure 10 fois par seconde (ex: capteur de température)
fs = 1/3600  → on mesure 1 fois par heure

Plus fs est grand, plus on capture les détails rapides du signal.
```

---
## 📝 Partie 2 – La FFT : voir les fréquences d'un signal

### 🔎 L'analogie du prisme

Un prisme décompose la lumière blanche en ses couleurs (rouge, vert, bleu…). La **FFT (Fast Fourier Transform)** fait la même chose avec un signal : elle le décompose en ses **fréquences constituantes**.

```
Signal complexe           FFT              Fréquences trouvées
(mélange de sons)   ─────────────→   Pic à 440 Hz  → note La
                                      Pic à 880 Hz  → octave
                                      Pic à 1320 Hz → tierce
```

**Cas d'usage concrets :**
- **Maintenance prédictive** : une machine vibre à une fréquence anormale → détection de panne
- **Analyse des ventes** : pics à 7 jours → cycle hebdomadaire, à 365 jours → saisonnalité annuelle
- **Médical** : analyse des battements du cœur (ECG)
- **Finance** : détecter des cycles dans les cours boursiers

### 🔎 Comment utiliser la FFT avec SciPy

```python
from scipy.fft import fft, fftfreq

# 1. Calculer la FFT
spectre   = fft(signal)              # transforme le signal
amplitude = np.abs(spectre) / len(signal)  # amplitude à chaque fréquence

# 2. Calculer les fréquences correspondantes
freqs = fftfreq(len(signal), d=1/fs)
#                              ↑
#                         intervalle entre deux mesures (1/fs)

# 3. Ne garder que les fréquences positives (la FFT est symétrique)
masque = freqs >= 0
freqs_pos = freqs[masque]
ampl_pos  = amplitude[masque]

# 4. Visualiser : les pics = les fréquences présentes dans le signal
plt.plot(freqs_pos, ampl_pos)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# Signal simple : une seule fréquence (10 Hz)
# Exemple : vibration d'une machine à 10 cycles/seconde
fs     = 500          # fréquence d'échantillonnage : 500 mesures/seconde
duree  = 2.0          # durée : 2 secondes
t      = np.linspace(0, duree, int(fs * duree), endpoint=False)

freq_signal = 10      # fréquence du signal : 10 Hz
signal      = np.sin(2 * np.pi * freq_signal * t)

# Calculer la FFT
spectre   = fft(signal)
amplitude = np.abs(spectre) / len(signal)
freqs     = fftfreq(len(signal), d=1/fs)

# Garder seulement les fréquences positives
masque    = freqs >= 0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))

# Signal dans le temps
ax1.plot(t[:100], signal[:100], color="steelblue", linewidth=1.5)
ax1.set_title("Signal temporel (vibration à 10 Hz) — zoom sur 0.2 secondes")
ax1.set_xlabel("Temps (s)")
ax1.set_ylabel("Amplitude")
ax1.grid(True, alpha=0.3)

# Spectre de fréquences
ax2.plot(freqs[masque], amplitude[masque], color="coral", linewidth=1.5)
ax2.axvline(freq_signal, color="red", linestyle="--",
            label=f"Fréquence détectée : {freq_signal} Hz")
ax2.set_title("Spectre de fréquences (FFT) — pic à 10 Hz")
ax2.set_xlabel("Fréquence (Hz)")
ax2.set_ylabel("Amplitude")
ax2.set_xlim(0, 50)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("→ La FFT détecte exactement le pic à 10 Hz — c'est la fréquence du signal")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# Signal complexe : mélange de 3 fréquences + bruit
# Exemple : capteur sur une machine avec plusieurs composantes de vibration
fs    = 1000
duree = 2.0
t     = np.linspace(0, duree, int(fs * duree), endpoint=False)

np.random.seed(42)
signal = (1.0 * np.sin(2 * np.pi * 5  * t) +   # composante à 5 Hz (forte)
          0.6 * np.sin(2 * np.pi * 20 * t) +   # composante à 20 Hz (moyenne)
          0.3 * np.sin(2 * np.pi * 50 * t) +   # composante à 50 Hz (faible)
          0.4 * np.random.randn(len(t)))         # bruit aléatoire

# FFT
spectre   = fft(signal)
amplitude = np.abs(spectre) / len(signal)
freqs     = fftfreq(len(signal), d=1/fs)
masque    = freqs >= 0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7))

ax1.plot(t[:200], signal[:200], color="steelblue", linewidth=1, alpha=0.8)
ax1.set_title("Signal complexe bruité (mélange de 5 Hz + 20 Hz + 50 Hz + bruit)")
ax1.set_xlabel("Temps (s)")
ax1.set_ylabel("Amplitude")
ax1.grid(True, alpha=0.3)

ax2.plot(freqs[masque], amplitude[masque], color="coral", linewidth=1.2)
for f, label in [(5, "5 Hz"), (20, "20 Hz"), (50, "50 Hz")]:
    ax2.axvline(f, color="navy", linestyle="--", alpha=0.7, label=label)
ax2.set_title("Spectre FFT — les 3 pics correspondent aux 3 fréquences du signal")
ax2.set_xlabel("Fréquence (Hz)")
ax2.set_ylabel("Amplitude")
ax2.set_xlim(0, 100)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("→ La FFT retrouve exactement les 3 fréquences malgré le bruit")

---
## 📝 Partie 3 – Filtrer un signal bruité

### 🔎 Qu'est-ce qu'un filtre ?

Un **filtre** laisse passer certaines fréquences et bloque les autres — comme un filtre à café qui retient le marc et laisse passer le liquide.

```
Filtre passe-bas  → garde les BASSES fréquences, supprime les hautes
                    Usage : lisser une courbe, éliminer un bruit haute fréquence

Filtre passe-haut → garde les HAUTES fréquences, supprime les basses
                    Usage : isoler les variations rapides, enlever la tendance

Filtre passe-bande → garde une PLAGE de fréquences
                     Usage : isoler un signal dans une bande précise
```

### 🔎 Appliquer un filtre avec SciPy

```python
from scipy.signal import butter, filtfilt

# Étape 1 : concevoir le filtre
b, a = butter(
    N    = 4,          # ordre du filtre (plus élevé = plus net)
    Wn   = 15,         # fréquence de coupure (Hz)
    btype= 'low',      # type : 'low', 'high', 'band'
    fs   = fs          # fréquence d'échantillonnage
)

# Étape 2 : appliquer le filtre au signal
signal_filtre = filtfilt(b, a, signal)
# filtfilt applique le filtre deux fois (aller + retour)
# pour éviter tout décalage temporel dans le résultat
```

> 💡 **`filtfilt` vs `lfilter`** : `filtfilt` est à préférer car il n'introduit pas de décalage. `lfilter` est plus rapide mais décale légèrement le signal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import butter, filtfilt

# Situation : capteur de température d'une machine
# Signal utile : variation lente de la température (1 Hz)
# Bruit : vibrations électriques parasites (haute fréquence)
np.random.seed(0)
fs    = 200
duree = 5.0
t     = np.linspace(0, duree, int(fs * duree), endpoint=False)

# Signal réel = tendance utile + bruit haute fréquence
tendance = 60 + 5 * np.sin(2 * np.pi * 0.5 * t)         # variation lente
bruit    = 2.0 * np.random.randn(len(t))                  # bruit aléatoire
signal_bruite = tendance + bruit

# Filtre passe-bas : garde les fréquences < 2 Hz (élimine le bruit)
b, a = butter(N=4, Wn=2, btype='low', fs=fs)
signal_filtre = filtfilt(b, a, signal_bruite)

fig, axes = plt.subplots(3, 1, figsize=(11, 10))

# Signal bruité
axes[0].plot(t, signal_bruite, color="gray", linewidth=0.8, alpha=0.8, label="Signal bruité")
axes[0].plot(t, tendance,      color="steelblue", linewidth=2, linestyle="--", label="Signal réel (caché)")
axes[0].set_title("Signal de température — capteur bruité")
axes[0].set_ylabel("Température (°C)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Signal filtré
axes[1].plot(t, signal_bruite, color="gray",      linewidth=0.8, alpha=0.5, label="Bruité")
axes[1].plot(t, signal_filtre, color="coral",     linewidth=2.5, label="Filtré (passe-bas 2Hz)")
axes[1].plot(t, tendance,      color="steelblue", linewidth=1.5, linestyle="--", label="Réel")
axes[1].set_title("Après filtrage — le bruit est éliminé")
axes[1].set_ylabel("Température (°C)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Spectres avant/après
def spectre_positif(sig, fs):
    sp = np.abs(fft(sig)) / len(sig)
    fr = fftfreq(len(sig), d=1/fs)
    mask = fr >= 0
    return fr[mask], sp[mask]

fr_b, sp_b = spectre_positif(signal_bruite, fs)
fr_f, sp_f = spectre_positif(signal_filtre, fs)

axes[2].plot(fr_b, sp_b, color="gray",  linewidth=1,   alpha=0.7, label="Avant filtrage")
axes[2].plot(fr_f, sp_f, color="coral", linewidth=2,   label="Après filtrage")
axes[2].axvline(2, color="navy", linestyle="--", label="Coupure : 2 Hz")
axes[2].set_title("Spectre FFT — le bruit haute fréquence est supprimé")
axes[2].set_xlabel("Fréquence (Hz)")
axes[2].set_ylabel("Amplitude")
axes[2].set_xlim(0, 30)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 📝 Partie 4 – Application métier : détecter les cycles dans des données

### 🔎 La FFT sur des données business

La FFT ne s'applique pas qu'aux sons et aux capteurs. Elle est très utile pour **détecter des cycles** dans des données temporelles business :

- **Ventes quotidiennes** → pic à 7 jours = cycle hebdomadaire
- **Trafic web horaire** → pic à 24h = cycle journalier
- **Appels service client** → pic à 5 jours = cycle semaine de travail

**Avantage :** la FFT détecte ces cycles même s'ils ne sont pas immédiatement visibles dans les données brutes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# Simulation : 90 jours de ventes quotidiennes
# Composantes : tendance de fond + cycle hebdomadaire + cycle mensuel + bruit
np.random.seed(7)
n_jours  = 90
jours    = np.arange(n_jours)

# Construction du signal de ventes
tendance    = 1000 + 5 * jours                          # croissance linéaire
cycle_hebdo = 200 * np.sin(2 * np.pi * jours / 7)      # cycle 7 jours
cycle_mensuel = 100 * np.sin(2 * np.pi * jours / 30)   # cycle 30 jours
bruit       = 80 * np.random.randn(n_jours)
ventes      = tendance + cycle_hebdo + cycle_mensuel + bruit

# FFT sur les ventes (sans la tendance — on la retire d'abord)
ventes_detrend = ventes - tendance   # retirer la tendance linéaire
spectre   = fft(ventes_detrend)
amplitude = np.abs(spectre) / n_jours
freqs     = fftfreq(n_jours, d=1)    # d=1 car 1 mesure par jour
# freqs sont en cycles/jour → période = 1/freq (en jours)

masque = freqs > 0   # seulement les fréquences positives

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(jours, ventes, color="steelblue", linewidth=1.5, label="Ventes réelles")
ax1.plot(jours, tendance, color="red", linestyle="--", linewidth=2, label="Tendance")
ax1.set_title("Ventes quotidiennes sur 90 jours")
ax1.set_xlabel("Jours")
ax1.set_ylabel("Ventes (€)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Axe en périodes (jours) plutôt qu'en fréquences
periodes = 1 / freqs[masque]   # conversion fréquence → période en jours
ax2.plot(periodes, amplitude[masque], color="coral", linewidth=1.5)
ax2.axvline(7,  color="navy",  linestyle="--", label="Cycle 7 jours (hebdo)")
ax2.axvline(30, color="green", linestyle="--", label="Cycle 30 jours (mensuel)")
ax2.set_title("Spectre FFT — cycles détectés dans les ventes")
ax2.set_xlabel("Période (jours)")
ax2.set_ylabel("Amplitude")
ax2.set_xlim(2, 50)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("→ La FFT détecte clairement les deux cycles : 7 jours (hebdomadaire) et ~30 jours (mensuel)")

---
## 🧩 Exercice 1 – Analyser les vibrations d'une machine

Un capteur de vibrations surveille une machine industrielle. Une vibration anormale à **25 Hz** indique un problème de roulement.

```python
np.random.seed(1)
fs    = 1000
duree = 3.0
t     = np.linspace(0, duree, int(fs * duree), endpoint=False)

# Signal : vibration normale (10 Hz) + anomalie (25 Hz) + bruit
signal = (0.8 * np.sin(2 * np.pi * 10 * t) +
          0.4 * np.sin(2 * np.pi * 25 * t) +
          0.3 * np.random.randn(len(t)))
```

1. Tracez le signal dans le temps (seulement les 200 premiers points)
2. Calculez et tracez le spectre FFT — identifiez les deux pics
3. Appliquez un **filtre passe-bas à 15 Hz** pour isoler la vibration normale
4. Appliquez un **filtre passe-haut à 15 Hz** pour isoler l'anomalie (25 Hz)
5. Affichez les deux signaux filtrés côte à côte

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import butter, filtfilt

np.random.seed(1)
fs    = 1000
duree = 3.0
t     = np.linspace(0, duree, int(fs * duree), endpoint=False)

signal = (0.8 * np.sin(2 * np.pi * 10 * t) +
          0.4 * np.sin(2 * np.pi * 25 * t) +
          0.3 * np.random.randn(len(t)))

# TODO 1 : tracer le signal (200 premiers points)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(t[:200], signal[:200], color="steelblue", linewidth=1)
axes[0, 0].set_title("Signal brut (vibrations machine)")
axes[0, 0].set_xlabel("Temps (s)")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].grid(True, alpha=0.3)

# TODO 2 : spectre FFT
spectre   = fft(signal)
amplitude = np.abs(spectre) / len(signal)
freqs     = fftfreq(len(signal), d=1/fs)
masque    = freqs >= 0

axes[0, 1].plot(freqs[masque], amplitude[masque], color="coral", linewidth=1.2)
axes[0, 1].axvline(10, color="navy",  linestyle="--", label="10 Hz (normal)")
axes[0, 1].axvline(25, color="red",   linestyle="--", label="25 Hz (anomalie)")
axes[0, 1].set_title("Spectre FFT")
axes[0, 1].set_xlabel("Fréquence (Hz)")
axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].set_xlim(0, 60)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# TODO 3 : filtre passe-bas (garde < 15 Hz)
b_low, a_low = butter(N=4, Wn=..., btype='low', fs=fs)
signal_normal = filtfilt(b_low, a_low, signal)

axes[1, 0].plot(t[:200], signal_normal[:200], color="seagreen", linewidth=1.5)
axes[1, 0].set_title("Filtré passe-bas (10 Hz) — vibration normale")
axes[1, 0].set_xlabel("Temps (s)")
axes[1, 0].set_ylabel("Amplitude")
axes[1, 0].grid(True, alpha=0.3)

# TODO 4 : filtre passe-haut (garde > 15 Hz)
b_high, a_high = butter(N=4, Wn=..., btype='high', fs=fs)
signal_anomalie = filtfilt(b_high, a_high, signal)

axes[1, 1].plot(t[:200], signal_anomalie[:200], color="crimson", linewidth=1.5)
axes[1, 1].set_title("Filtré passe-haut (25 Hz) — anomalie isolée")
axes[1, 1].set_xlabel("Temps (s)")
axes[1, 1].set_ylabel("Amplitude")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("Analyse de vibrations — maintenance prédictive", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

<details>
<summary>💡 Correction</summary>

```python
# Les seules lignes à compléter :

# TODO 3 : filtre passe-bas à 15 Hz
b_low, a_low = butter(N=4, Wn=15, btype='low', fs=fs)
signal_normal = filtfilt(b_low, a_low, signal)
# → isole la vibration à 10 Hz (fonctionnement normal)

# TODO 4 : filtre passe-haut à 15 Hz
b_high, a_high = butter(N=4, Wn=15, btype='high', fs=fs)
signal_anomalie = filtfilt(b_high, a_high, signal)
# → isole la vibration à 25 Hz (anomalie de roulement)

# Interprétation :
# - Le signal filtré passe-bas montre la vibration normale à 10 Hz
# - Le signal filtré passe-haut révèle l'anomalie à 25 Hz
# - En production : si l'amplitude de 25 Hz dépasse un seuil → alerte maintenance
```
</details>

---
## 🧩 Exercice 2 – Détecter des cycles dans des données de ventes

Vous avez 180 jours de données de ventes quotidiennes d'un e-commerce :

```python
np.random.seed(42)
jours = np.arange(180)
ventes = (2000
          + 500  * np.sin(2 * np.pi * jours / 7)    # cycle hebdo
          + 300  * np.sin(2 * np.pi * jours / 30)   # cycle mensuel
          + 150  * np.random.randn(180))             # bruit
```

1. Tracez les ventes sur 180 jours
2. Appliquez la FFT et tracez le spectre en **périodes** (jours) — pas en fréquences
3. Identifiez les deux pics principaux (7 jours et 30 jours)
4. Appliquez un **filtre passe-bas** (fréquence de coupure : 1/14 cycle/jour) pour lisser les ventes et voir la tendance
5. Affichez les ventes brutes et lissées sur le même graphique

> 💡 **Indice pour la fréquence de coupure :** avec `fs=1` (1 mesure/jour), une coupure à `1/14` garde les cycles > 14 jours et élimine les variations hebdomadaires.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import butter, filtfilt

np.random.seed(42)
jours  = np.arange(180)
ventes = (2000
          + 500  * np.sin(2 * np.pi * jours / 7)
          + 300  * np.sin(2 * np.pi * jours / 30)
          + 150  * np.random.randn(180))

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# TODO 1 : ventes brutes
axes[0].plot(jours, ventes, color="steelblue", linewidth=1.2)
axes[0].set_title("Ventes quotidiennes sur 180 jours")
axes[0].set_xlabel("Jours")
axes[0].set_ylabel("Ventes (€)")
axes[0].grid(True, alpha=0.3)

# TODO 2 : FFT + spectre en périodes
fs_data   = 1   # 1 mesure par jour
spectre   = fft(ventes - ventes.mean())   # centrer les données d'abord
amplitude = np.abs(spectre) / len(ventes)
freqs     = fftfreq(len(ventes), d=1/fs_data)
masque    = freqs > 0
periodes  = 1 / freqs[masque]

axes[1].plot(periodes, amplitude[masque], color="coral", linewidth=1.5)
# TODO 3 : annoter les pics
axes[1].axvline(7,  color="navy",  linestyle="--", label="7 jours (hebdomadaire)")
axes[1].axvline(30, color="green", linestyle="--", label="30 jours (mensuel)")
axes[1].set_title("Spectre FFT — cycles détectés")
axes[1].set_xlabel("Période (jours)")
axes[1].set_ylabel("Amplitude")
axes[1].set_xlim(2, 60)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# TODO 4 : filtre passe-bas pour lisser
b, a = butter(N=3, Wn=..., btype='low', fs=fs_data)
ventes_lissees = filtfilt(b, a, ventes)

# TODO 5 : brut + lissé
axes[2].plot(jours, ventes,         color="steelblue", linewidth=0.8, alpha=0.5, label="Ventes brutes")
axes[2].plot(jours, ventes_lissees, color="red",       linewidth=2.5, label="Tendance lissée")
axes[2].set_title("Ventes brutes vs tendance lissée")
axes[2].set_xlabel("Jours")
axes[2].set_ylabel("Ventes (€)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<details>
<summary>💡 Correction</summary>

```python
# La seule ligne à compléter :

# TODO 4 : fréquence de coupure = 1/14 cycle/jour
# → garde les cycles plus longs que 14 jours
# → élimine le cycle hebdomadaire (7 jours) et le bruit
b, a = butter(N=3, Wn=1/14, btype='low', fs=1)
ventes_lissees = filtfilt(b, a, ventes)

# Résultat :
# - Le spectre montre clairement 2 pics : à 7 jours et à 30 jours
# - La courbe lissée montre la tendance mensuelle sans le "bruit" hebdomadaire
# - Utile pour le reporting mensuel : présenter la tendance de fond
#   sans être perturbé par les variations jour-à-jour
```
</details>

---
## ✅ Récapitulatif

### La FFT

| Concept | Code | Ce qu'il faut retenir |
|---------|------|-----------------------|
| **Calculer la FFT** | `spectre = fft(signal)` | Décompose le signal en fréquences |
| **Amplitude** | `np.abs(spectre) / len(signal)` | Intensité de chaque fréquence |
| **Fréquences** | `fftfreq(len(signal), d=1/fs)` | Axe des fréquences correspondant |
| **Fréquences positives** | `masque = freqs >= 0` | La FFT est symétrique — on ne garde que la moitié |
| **Périodes** | `periodes = 1 / freqs[masque]` | Convertir fréquences en jours/secondes |

### Le filtrage

| Type de filtre | `btype=` | Garde | Élimine |
|---------------|----------|-------|--------|
| **Passe-bas** | `'low'` | Basses fréquences (variations lentes) | Bruit haute fréquence |
| **Passe-haut** | `'high'` | Hautes fréquences (variations rapides) | Tendance de fond |
| **Passe-bande** | `'band'` | Une plage de fréquences | Le reste |

```python
b, a          = butter(N=4, Wn=coupure, btype='low', fs=fs)
signal_filtre = filtfilt(b, a, signal)
```

**Cas d'usage en entreprise :**
- **Maintenance prédictive** : détecter une fréquence anormale dans les vibrations d'une machine
- **Analyse des ventes** : identifier les cycles hebdomadaires et saisonniers
- **Lissage de courbes** : présenter une tendance claire sans le bruit journalier
- **Détection d'anomalies** : une fréquence apparaît soudainement → anomalie à investiguer

---
📘 **Module SciPy terminé — Bravo ! 🎉**